<a href="https://colab.research.google.com/github/TreinamentoICCInatel/CursoVW/blob/main/EDA/Clusteriza%C3%A7%C3%A3o.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


Importando as bibliotecas



In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

Carregando os dados do CSV que está no Github

In [ ]:
data = pd.read_csv('https://raw.githubusercontent.com/TreinamentoICCInatel/CursoVW/refs/heads/main/EDA/Auto%20Sales%20data.csv')

Criando um novo DataFrame (rfm) agrupado por CUSTOMERNAME e agregado com as outras 3 colunas - dias, frequência e valor total de pedidos.

In [ ]:
rfm = (
    data.groupby("CUSTOMERNAME")
    .agg(
        {
            "DAYS_SINCE_LASTORDER": "min",
            "ORDERNUMBER": "nunique",
            "SALES": "sum",
        }
    )
    .reset_index()
)

In [ ]:
rfm.head()

Renomeando as colunas para melhor identificação

In [ ]:
rfm.columns = ["CUSTOMERNAME", "Recorrência", "Frequência", "Monetário"]

In [ ]:
rfm.head()

Aplicando uma transformação nos dados, para ficar com média 0 e desvio padrão 1

In [ ]:
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[["Recorrência", "Frequência", "Monetário"]])

In [ ]:
rfm_scaled

Aplica PCA - Principal Component Analysis para redução de dimensionalidade nos dados. Reduz complexidade computacional e ajuda a melhorar o desempenho da Clusterização

In [ ]:
pca = PCA(n_components=0.95, random_state=42)
rfm_pca = pca.fit_transform(rfm_scaled)

In [ ]:
n_clusters_range = range(2, 11)

In [ ]:
best_score = -1
best_n_clusters  = None

Executa o algoritmo KMeans na faixa de valores definido anteriormento para encontar o melhor agrupamento possível. É uma espécie de GridSearch.

In [ ]:
for n_clusters in n_clusters_range:
  kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
  clusters = kmeans.fit_predict(rfm_pca)
  score = silhouette_score(rfm_pca, clusters)
  if score > best_score:
    best_score = score
    best_n_clusters = n_clusters

Com o melhor valor de Quantidade de Cluster definido no GridSearch, executa novamente o KMeans com este valor.

In [ ]:
kmean = KMeans(n_clusters=best_n_clusters, random_state=42, n_init=10)

Usa o dados após PCA para ajustar o modelo não supervisionado, encontrando e associando os clusters aos respectivos dados. Uma nova coluna é adicionada ao DataFrame original rfm.

In [ ]:
rfm["Cluster"] = kmeans.fit_predict(rfm_pca)

Cálculo do Score

In [ ]:
silhouette_avg = silhouette_score(rfm_pca, rfm["Cluster"])

Criação de um novo DataFrame somente com nome do cliente e o Cluster ajustado pelo KMenans

In [ ]:
submission = pd.DataFrame({"CUSTOMERNAME": rfm["CUSTOMERNAME"], "Cluster" : rfm["Cluster"]})

In [ ]:
submission

**VISUALIZAÇÕES**

In [ ]:
submission["CUSTOMERNAME"] = submission["CUSTOMERNAME"].astype("category")

y = [1] * len(submission)

plt.figure(figsize=(14, 4))
scatter = plt.scatter(
    submission["CUSTOMERNAME"],
    y,
    c=submission["Cluster"],       # cores por cluster
    cmap="bwr",            # azul/vermelho (0/1), mas pode trocar
    s=100,                 # tamanho do ponto
    edgecolors="black"
)

plt.title("Clientes por Cluster", fontsize=7)
plt.yticks([])
plt.xticks(rotation=65, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(14, 6))
sns.swarmplot(
    data=submission,
    x="CUSTOMERNAME",
    y="Cluster",
    hue="Cluster",
    palette="tab10"
)

plt.xticks(rotation=65, ha="right")
plt.title("Distribuição de Clientes por Cluster (Swarmplot)")
plt.tight_layout()
plt.show()


In [ ]:
rfm.query('Cluster == 2')

In [ ]:
plt.figure(figsize=(12, 8))
cmap = plt.get_cmap("tab10")

scatter = plt.scatter(
    rfm["Frequency"],
    rfm["Monetary"],
    c=rfm["Cluster"],
    cmap=cmap,
    s=20,
    alpha=0.8,
)

plt.xlabel("Frequency", fontsize=14)
plt.ylabel("Monetary", fontsize=14)
plt.title("Clusters de Clientes (Frequency x Monetary)", fontsize=16)
plt.xlim(0, 10)
plt.ylim(0, 300000)

plt.grid(True, linestyle="--", alpha=0.8)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
cmap = plt.get_cmap("tab10")

scatter = plt.scatter(
    rfm["Recency"],
    rfm["Monetary"],
    c=rfm["Cluster"],
    cmap=cmap,
    s=20,
    alpha=0.8,
)

plt.xlabel("Recency", fontsize=14)
plt.ylabel("Monetary", fontsize=14)
plt.title("Clusters de Clientes (Recency x Monetary)", fontsize=16)

plt.grid(True, linestyle="--", alpha=0.8)
plt.tight_layout()
plt.show()
